## InMemoryChatMessageHistory

In [2]:
from langchain_classic.chains.summarize.refine_prompts import prompt_template
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder

#实例化
history = InMemoryChatMessageHistory()
#添加相关的消息存储
history.add_user_message("你好")

history.add_ai_message("很高兴认识你！")

print(history.messages)
print(type(history.messages))##<class 'list'>

[HumanMessage(content='你好', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你！', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
<class 'list'>


## 结合大模型

In [6]:
from langchain_openai import ChatOpenAI
import os
import dotenv
from langchain_classic.chains import LLMChain
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import MessagesPlaceholder,ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


dotenv.load_dotenv()
llm=ChatOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    temperature=0,
    model="deepseek-chat",
)
prompt_template=ChatPromptTemplate.from_messages(
    [
        ("system", "你的名字叫小丽，是一个AI助手。"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)
history = InMemoryChatMessageHistory()

# chain = prompt_template | llm | StrOutputParser()
chain = prompt_template | llm

def chat(user_input: str):
    result = chain.invoke(
        {
            "input": user_input,
            "history": history.messages,
        }
    )
    history.add_user_message(user_input)
    history.add_ai_message(result)
    return result

print(chat("我叫小林"))
print("########################")
print(chat("你叫什么名字？"))
print("########################")
print(chat("我叫什么名字？"))

content='你好小林！很高兴认识你！😊 有什么我可以帮助你的吗？' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 16, 'total_tokens': 32, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 16}, 'model_provider': 'openai', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410', 'id': '40d7a8a3-54ed-40a5-a568-c58230b68f83', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d8751-df67-7342-9b18-71737f6eb3f7-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 16, 'output_tokens': 16, 'total_tokens': 32, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}
########################
content='我叫小丽，是你的AI助手！有什么问题或者需要帮忙的地方，尽管告诉我哦～ 😊' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 40, 'tota

## RunnableWithMessageHistory结合大模型

In [4]:
from langchain_core.runnables import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_openai import ChatOpenAI
import os
import dotenv

dotenv.load_dotenv()
llm=ChatOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    temperature=0,
    model="deepseek-chat",
)
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个有记忆能力的AI助手，名字叫小丽。"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# 3. 输出解析器
parser = StrOutputParser()

# 4. 普通链
chain = prompt | llm | parser

# 5. 用字典保存不同 session 的聊天历史
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 6. 包装成带历史记录的链
with_history = RunnableWithMessageHistory(
    runnable=chain,
    get_session_history=get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# -------------------------
# 第一次对话
# -------------------------
response1 = with_history.invoke(
    {"input": "我叫小林。"},
    config={"configurable": {"session_id": "1"}}   # 写死为 1
)
print("第一轮回答：", response1)

# -------------------------
# 第二次对话
# -------------------------
response2 = with_history.invoke(
    {"input": "我叫什么名字？"},
    config={"configurable": {"session_id": "1"}}   # 仍然写死为 1
)
print("第二轮回答：", response2)

第一轮回答： 小林你好！很高兴认识你！我是小丽，有什么我可以帮你的吗？😊
第二轮回答： 你刚才告诉我你叫小林呀！我会记住的，小林～ 😄


## ConversationKGMemory 基于知识图谱的对话记忆模块

In [3]:
from langchain_community.memory.kg import ConversationKGMemory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_openai import ChatOpenAI
import os
import dotenv

dotenv.load_dotenv()
llm=ChatOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    temperature=0,
    model="deepseek-chat",
)

memory=ConversationKGMemory(llm=llm)

#保存对象
memory.save_context({"input": "向小丽问好"}, {"output": "小丽是谁"})
memory.save_context({"input": "小丽是我的女朋友"}, {"output": "好的"})
#查询会话
#返回的并不是聊天机器人最终要显示给用户的回答，而是准备塞进 prompt 的辅助上下文
memory.load_memory_variables({"input": "小丽是谁"})

{'history': 'On 小丽: 小丽 是 我的女朋友.'}

In [4]:
memory.get_knowledge_triplets("她最喜欢咬人")#把一段文本里的事实关系抽取成“知识三元组”列表

[KnowledgeTriple(subject='小丽', predicate='最喜欢', object_='咬人')]